# Mistral Fine-tuning API

Check out the docs: https://docs.mistral.ai/capabilities/finetuning/

In [30]:
# !pip install mistralai pandas
with open("data/secrets/mistral.secret") as f:
    os.environ["MISTRAL_API_KEY"] = f.read()
    api_key = os.environ["MISTRAL_API_KEY"]

## Prepare the dataset

In this example, let’s use the ultrachat_200k dataset. We load a chunk of the data into Pandas Dataframes, split the data into training and validation, and save the data into the required jsonl format for fine-tuning.

In [2]:
import pandas as pd
df = pd.read_parquet('https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k/resolve/main/data/test_gen-00000-of-00001-3d4cd8309148a71f.parquet')
df



,prompt,prompt_id,messages
24636,Read the passage below and answer the question...,cef194424ecca27ac831d66361160cab4d43dfe297b6df...,[{'content': 'Read the passage below and answe...
10445,Create a comprehensive checklist that includes...,6b8e5334656624d8632490924930514d89f65712bc270d...,[{'content': 'Create a comprehensive checklist...
24929,Can you summarize the controversy surrounding ...,0dffb0a7d5cf4496409b638815d255773b79a618f36a4e...,[{'content': 'Can you summarize the controvers...
6576,Create a set of ten posters that raise awarene...,23bbd05fabd4a47c9485d9e1a7f3589ac67b5367a4044d...,[{'content': 'Create a set of ten posters that...
544,Write clear and concise instructions for how t...,0785d5c6529eaeaa2ae46334592d7b1faa981e7dcae65f...,[{'content': 'Write clear and concise instruct...
...,...,...,...
22134,Can you paraphrase the text about Viktor & Rol...,b9ec7da5bba36dd2cefd2d27db763ebe3ea3c25cb8541f...,[{'content': 'Can you paraphrase the text abou...
23722,Write step-by-step instructions on how to make...,405861102893c1f3fe7d005f9965a5dd658fe5a9f3ebfd...,[{'content': 'Write step-by-step instructions ...
4731,Write a horror story in a gothic style featuri...,fc015c9dd96b45ba4088a95680fbf7d63ae9fabd035d71...,[{'content': 'Write a horror story in a gothic...
15909,Is it possible to obtain a translated version ...,ada9e8528bc772ead5a169af3d66f4e873223f2181239d...,[{'content': 'Is it possible to obtain a trans...


In [25]:
df_train=df[:100]
df_eval=df[100:200]

# df_train.to_json("data/ultrachat_chunk_train.jsonl", orient="records", lines=True)
# df_eval.to_json("data/ultrachat_chunk_eval.jsonl", orient="records", lines=True)
df_eval

,prompt,prompt_id,messages
100,Please write a well-researched and informative...,f93b4789593342c90018ed42fc2f8889cbe00de938ac93...,[{'content': 'Please write a well-researched a...
101,What are the main features offered by air mile...,450104688d133b3b25606cdac0e14cdbc30688fe2b3dd1...,[{'content': 'What are the main features offer...
102,"Let’s go shroomin’, Daddy chortles. of our paw...",4c53368fb13c7d012ad89535856ad6199d118146910150...,"[{'content': 'Let’s go shroomin’, Daddy chortl..."
103,Generate response to the question/instruction ...,badc2b4b1dfec2318648c2c281719df8eaad42b9e248e7...,[{'content': 'Generate response to the questio...
104,Write a 500-word short story in third person n...,d10fb0a6682f0874fc035e988f323b488b3d65f9f988da...,[{'content': 'Write a 500-word short story in ...
...,...,...,...
195,Here is a piece of text: A joke or single reli...,81e09b61ff79bda207c9264c945acb719c41d4201d42ca...,[{'content': 'Here is a piece of text: A joke ...
196,How did the marine engine and boat segments of...,8bf1d793af00ca6df29f9e5d58b897f5093206c4c9fb4a...,[{'content': 'How did the marine engine and bo...
197,Create a comprehensive blog post about time-sa...,ce48ba347744cd10a8215ce2cbf045527cd3b013cccd82...,[{'content': 'Create a comprehensive blog post...
198,Write a well-structured and comprehensive essa...,313d4e18ef7e004f585ec2aa35df2cc4ae15ffd160e8f6...,[{'content': 'Write a well-structured and comp...


## Reformat dataset
If you upload this ultrachat_chunk_train.jsonl to Mistral API, you might encounter an error message “Invalid file format” due to data formatting issues. To reformat the data into the correct format, you can download the reformat_dataset.py script and use it to validate and reformat both the training and evaluation data:

In [18]:
# download the validation and reformat script
# !wget https://raw.githubusercontent.com/mistralai/mistral-finetune/main/utils/reformat_data.py

In [26]:
# validate and reformat the training data
# !python reformat_data.py data/ultrachat_chunk_train.jsonl

In [27]:
# validate the reformat the eval data
# !python reformat_data.py data/ultrachat_chunk_eval.jsonl

In [28]:
df_train.iloc[55]['messages']

array([{'content': 'Write a plot outline for a fast-paced thriller in which an experienced spy races against time to prevent a group of terrorists from carrying out a devastating attack. Ensure that your story features plenty of heart-stopping action, high-tech espionage, and unexpected plot twists to keep readers on the edge of their seats from beginning to end. Consider incorporating elements of political intrigue, international espionage, and personal stake in the hero’s success to create a truly immersive and engaging reading experience.', 'role': 'user'},
       {'content': 'Title: The Final Countdown\n\nAct 1:\n- Introduction of protagonist, John Steele, a highly skilled and experienced spy, who is called into action once again\n- A group of terrorists led by a mysterious mastermind, known only as "The Shadow", plan to carry out a devastating attack on an unknown location and Steele is tasked with preventing it\n- Steele discovers that the target of the attack is a major metropol

## Upload dataset

In [32]:
from mistralai import Mistral
import os

api_key = os.environ["MISTRAL_API_KEY"]

client = Mistral(api_key=api_key)

ultrachat_chunk_train = client.files.upload(file={
    "file_name": "data/ultrachat_chunk_train.jsonl",
    "content": open("data/ultrachat_chunk_train.jsonl", "rb"),
})
ultrachat_chunk_eval = client.files.upload(file={
    "file_name": "data/ultrachat_chunk_eval.jsonl",
    "content": open("data/ultrachat_chunk_eval.jsonl", "rb"),
})

In [33]:
import json
def pprint(obj):
    print(json.dumps(obj.dict(), indent=4))

In [34]:
pprint(ultrachat_chunk_train)

{
    "id": "79eeb1bd-ce96-477c-8f37-80bfe0ab03f9",
    "object": "file",
    "bytes": null,
    "created_at": 1760922442,
    "filename": "ultrachat_chunk_train.jsonl",
    "purpose": "fine-tune",
    "sample_type": "instruct",
    "source": "upload",
    "num_lines": 100,
    "mimetype": "application/jsonl",
    "signature": "fb43e507a74da1c9"
}


C:\Users\conta\AppData\Local\Temp\ipykernel_33524\3084144206.py:3: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(json.dumps(obj.dict(), indent=4))


In [35]:
pprint(ultrachat_chunk_eval)

{
    "id": "19ac51de-b3da-4c6b-8786-d27c24d9eacc",
    "object": "file",
    "bytes": null,
    "created_at": 1760922459,
    "filename": "ultrachat_chunk_eval.jsonl",
    "purpose": "fine-tune",
    "sample_type": "instruct",
    "source": "upload",
    "num_lines": 100,
    "mimetype": "application/jsonl",
    "signature": "973b174524a0f4d9"
}


C:\Users\conta\AppData\Local\Temp\ipykernel_33524\3084144206.py:3: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(json.dumps(obj.dict(), indent=4))


## Create a fine-tuning job

In [ ]:
created_jobs = client.fine_tuning.jobs.create(
    model="open-mistral-7b",
    training_files=[{"file_id": ultrachat_chunk_train.id, "weight": 1}],
    validation_files=[ultrachat_chunk_eval.id],
    hyperparameters={
    "training_steps": 2,
    "learning_rate":0.0001
    },
    auto_start=True
)
created_jobs

JobOut(id='20178c3c-d75b-428e-b20d-7d39aa2b7468', auto_start=True, hyperparameters=TrainingParameters(training_steps=10, learning_rate=0.0001, epochs=None, fim_ratio=None), model='open-mistral-7b', status='QUEUED', job_type='FT', created_at=1721405548, modified_at=1721405548, training_files=['ec5af16a-77fe-4e14-ad09-47ead2848ce6'], validation_files=['d0c643a2-a57c-4031-bda7-5c9d6c3ec3e4'], OBJECT='job', fine_tuned_model=None, suffix=None, integrations=[], trained_tokens=None, repositories=[], metadata=JobMetadataOut(expected_duration_seconds=None, cost=None, cost_currency=None, train_tokens_per_step=None, train_tokens=None, data_tokens=None, estimated_start_time=None))

In [ ]:
pprint(created_jobs)

{
    "id": "2c002adb-12be-4a5d-a3ef-baacf9025be9",
    "auto_start": true,
    "hyperparameters": {
        "training_steps": 10,
        "learning_rate": 0.0001,
        "epochs": null,
        "fim_ratio": null
    },
    "model": "open-mistral-7b",
    "status": "QUEUED",
    "job_type": "FT",
    "created_at": 1721405164,
    "modified_at": 1721405164,
    "training_files": [
        "da7305f6-d649-4fe2-8132-d6f17c7c8141"
    ],
    "validation_files": [
        "f33adf06-8d79-420f-8722-d988cd60fbd6"
    ],
    "fine_tuned_model": null,
    "suffix": null,
    "integrations": [],
    "trained_tokens": null,
    "repositories": [],
    "metadata": {
        "expected_duration_seconds": null,
        "cost": null,
        "cost_currency": null,
        "train_tokens_per_step": null,
        "train_tokens": null,
        "data_tokens": null,
        "estimated_start_time": null
    }
}


In [ ]:
jobs = client.fine_tuning.jobs.list()
print(jobs)

total=32 data=[JobOut(id='20178c3c-d75b-428e-b20d-7d39aa2b7468', auto_start=True, hyperparameters=TrainingParameters(training_steps=10, learning_rate=0.0001, epochs=0.0431941570306258, fim_ratio=None), model='open-mistral-7b', status='RUNNING', job_type='FT', created_at=1721405548, modified_at=1721405549, training_files=['ec5af16a-77fe-4e14-ad09-47ead2848ce6'], validation_files=['d0c643a2-a57c-4031-bda7-5c9d6c3ec3e4'], OBJECT='job', fine_tuned_model=None, suffix=None, integrations=[], trained_tokens=None, repositories=[], metadata=JobMetadataOut(expected_duration_seconds=120, cost=2.6214, cost_currency='USD', train_tokens_per_step=131072, train_tokens=1310720, data_tokens=30344845, estimated_start_time=None)), JobOut(id='98cbe43b-61e5-47a8-a5fb-57f6d82abc0a', auto_start=True, hyperparameters=TrainingParameters(training_steps=10, learning_rate=0.0001, epochs=0.0431941570306258, fim_ratio=None), model='open-mistral-7b', status='RUNNING', job_type='FT', created_at=1721405463, modified_at=

In [ ]:
retrieved_jobs = client.fine_tuning.jobs.get(job_id = created_jobs.id)
retrieved_jobs

DetailedJobOut(id='20178c3c-d75b-428e-b20d-7d39aa2b7468', auto_start=True, hyperparameters=TrainingParameters(training_steps=10, learning_rate=0.0001, epochs=0.0431941570306258, fim_ratio=None), model='open-mistral-7b', status='RUNNING', job_type='FT', created_at=1721405548, modified_at=1721405549, training_files=['ec5af16a-77fe-4e14-ad09-47ead2848ce6'], validation_files=['d0c643a2-a57c-4031-bda7-5c9d6c3ec3e4'], OBJECT='job', fine_tuned_model=None, suffix=None, integrations=[], trained_tokens=None, repositories=[], metadata=JobMetadataOut(expected_duration_seconds=120, cost=2.6214, cost_currency='USD', train_tokens_per_step=131072, train_tokens=1310720, data_tokens=30344845, estimated_start_time=None), events=[EventOut(name='status-updated', created_at=1721405549, data=Unset()), EventOut(name='status-updated', created_at=1721405549, data=Unset()), EventOut(name='status-updated', created_at=1721405549, data=Unset()), EventOut(name='status-updated', created_at=1721405548, data=Unset()), 

In [ ]:
import time

retrieved_job = client.fine_tuning.jobs.get(job_id = created_jobs.id)
while retrieved_job.status in ["RUNNING", "QUEUED"]:
    retrieved_job = client.fine_tuning.jobs.get(job_id = created_jobs.id)
    pprint(retrieved_job)
    print(f"Job is {retrieved_job.status}, waiting 10 seconds")
    time.sleep(10)



{
    "id": "20178c3c-d75b-428e-b20d-7d39aa2b7468",
    "auto_start": true,
    "hyperparameters": {
        "training_steps": 10,
        "learning_rate": 0.0001,
        "epochs": 0.0431941570306258,
        "fim_ratio": null
    },
    "model": "open-mistral-7b",
    "status": "RUNNING",
    "job_type": "FT",
    "created_at": 1721405548,
    "modified_at": 1721405549,
    "training_files": [
        "ec5af16a-77fe-4e14-ad09-47ead2848ce6"
    ],
    "validation_files": [
        "d0c643a2-a57c-4031-bda7-5c9d6c3ec3e4"
    ],
    "fine_tuned_model": null,
    "suffix": null,
    "integrations": [],
    "trained_tokens": null,
    "repositories": [],
    "metadata": {
        "expected_duration_seconds": 120,
        "cost": 2.6214,
        "cost_currency": "USD",
        "train_tokens_per_step": 131072,
        "train_tokens": 1310720,
        "data_tokens": 30344845,
        "estimated_start_time": null
    },
    "events": [
        {
            "name": "status-updated",
          

In [ ]:
# List jobs
jobs = client.fine_tuning.jobs.list()
pprint(jobs)

{
    "total": 32,
    "data": [
        {
            "id": "20178c3c-d75b-428e-b20d-7d39aa2b7468",
            "auto_start": true,
            "hyperparameters": {
                "training_steps": 10,
                "learning_rate": 0.0001,
                "epochs": 0.0431941570306258,
                "fim_ratio": null
            },
            "model": "open-mistral-7b",
            "status": "SUCCESS",
            "job_type": "FT",
            "created_at": 1721405548,
            "modified_at": 1721405693,
            "training_files": [
                "ec5af16a-77fe-4e14-ad09-47ead2848ce6"
            ],
            "validation_files": [
                "d0c643a2-a57c-4031-bda7-5c9d6c3ec3e4"
            ],
            "fine_tuned_model": "ft:open-mistral-7b:b6e34a5e:20240719:20178c3c",
            "suffix": null,
            "integrations": [],
            "trained_tokens": 1310720,
            "repositories": [],
            "metadata": {
                "expected_duration_s

In [ ]:
# Retrieve a jobs
retrieved_jobs = client.fine_tuning.jobs.get(job_id = created_jobs.id)
pprint(retrieved_jobs)


{
    "id": "20178c3c-d75b-428e-b20d-7d39aa2b7468",
    "auto_start": true,
    "hyperparameters": {
        "training_steps": 10,
        "learning_rate": 0.0001,
        "epochs": 0.0431941570306258,
        "fim_ratio": null
    },
    "model": "open-mistral-7b",
    "status": "SUCCESS",
    "job_type": "FT",
    "created_at": 1721405548,
    "modified_at": 1721405693,
    "training_files": [
        "ec5af16a-77fe-4e14-ad09-47ead2848ce6"
    ],
    "validation_files": [
        "d0c643a2-a57c-4031-bda7-5c9d6c3ec3e4"
    ],
    "fine_tuned_model": "ft:open-mistral-7b:b6e34a5e:20240719:20178c3c",
    "suffix": null,
    "integrations": [],
    "trained_tokens": 1310720,
    "repositories": [],
    "metadata": {
        "expected_duration_seconds": 120,
        "cost": 2.6214,
        "cost_currency": "USD",
        "train_tokens_per_step": 131072,
        "train_tokens": 1310720,
        "data_tokens": 30344845,
        "estimated_start_time": null
    },
    "events": [
        {
  

## Use a fine-tuned model

In [ ]:
chat_response = client.chat.complete(
    model = retrieved_jobs.fine_tuned_model,
    messages = [{"role":'user', "content":'What is the best French cheese?'}]
)

In [ ]:
pprint(chat_response)

{
    "id": "1fac96713fd74799922712e34e009f81",
    "object": "chat.completion",
    "model": "ft:open-mistral-7b:b6e34a5e:20240719:20178c3c",
    "usage": {
        "prompt_tokens": 10,
        "completion_tokens": 73,
        "total_tokens": 83
    },
    "created": 1721405725,
    "choices": [
        {
            "index": 0,
            "finish_reason": "stop",
            "message": {
                "content": "There isn't a single \"best\" French cheese as there are hundreds of different types of cheese to choose from, each with its unique taste and texture. Some popular French cheeses include Brie, Camembert, Roquefort, Comt\u00e9, and Ch\u00e8vre. Try different cheeses to find out the one you like best!",
                "tool_calls": null,
                "prefix": false,
                "role": "assistant"
            }
        }
    ]
}


## Integration with Weights and Biases
We can also offer support for integration with Weights & Biases (W&B) to monitor and track various metrics and statistics associated with our fine-tuning jobs. To enable integration with W&B, you will need to create an account with W&B and add your W&B information in the “integrations” section in the job creation request:



In [ ]:
client.fine_tuning.jobs.create(
    model="open-mistral-7b",
    training_files=[{"file_id": ultrachat_chunk_train.id, "weight": 1}],
    validation_files=[ultrachat_chunk_eval.id],
    hyperparameters={"training_steps": 10, "learning_rate": 0.0001},
    integrations=[
        {
            "project": "<value>",
            "api_key": "<value>",
        }
    ]
)